# TIR → RGB Pipeline — Colab Driver (BAH 2026 · PS10)
Thin driver notebook: all real logic lives in the repo (`src/`). Run cells top-to-bottom.

**Stages:** setup → data download (GEE) → patch prep → train colorization → train SR → evaluate → demo.

> Long training runs go on **Kaggle** (30 h/week GPU); this notebook has bundle/restore cells for that workflow.

In [ ]:
#@title 1 · Setup (run in every fresh runtime)
import os
REPO = 'https://github.com/vighaneshgursale/tir2rgb-bah2026.git'

if not os.path.isdir('/content/tir2rgb-bah2026'):
    !git clone $REPO /content/tir2rgb-bah2026
%cd /content/tir2rgb-bah2026
!pip install -q -r requirements.txt

# basicsr / new-torchvision compatibility patch (known Colab gotcha)
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' \
    $(python -c "import basicsr, os; print(os.path.join(os.path.dirname(basicsr.__file__),'data','degradations.py'))")

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — Runtime > Change runtime type > GPU')

In [ ]:
#@title 2 · Mount Drive (checkpoints + dataset backups live here)
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/BAH2026'
import os; os.makedirs(DRIVE, exist_ok=True)

In [ ]:
#@title 3 · Download raw tiles from Google Earth Engine (~30 cities × 3×3 grid)
GEE_PROJECT = 'hackathonisro2026'  #@param {type:"string"}

import ee
ee.Authenticate()

# restore any previously downloaded tiles first (resumable)
![ -f $DRIVE/raw_tiles.zip ] && unzip -qn $DRIVE/raw_tiles.zip -d .
!python -m src.data.download_gee --project $GEE_PROJECT --out data/raw --grid 3

# back up to Drive
!cd data && zip -qr /content/raw_tiles.zip raw && cp /content/raw_tiles.zip $DRIVE/
print('tiles:', len(os.listdir('data/raw')))

In [ ]:
#@title 4 · Build training sets for both stages
!python -m src.data.prepare_patches --raw data/raw --out data

# --- alignment sanity check: overlay 3 random pairs (MUST look aligned) ---
import glob, random, numpy as np, matplotlib.pyplot as plt
from PIL import Image
samples = random.sample(glob.glob('data/tir2rgb/train/*.png'), 3)
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for ax_row, p in zip(axes, samples):
    ab = np.array(Image.open(p)); a, b = ab[:, :256, 0], ab[:, 256:]
    ax_row[0].imshow(a, cmap='gray'); ax_row[0].set_title('TIR 100m')
    ax_row[1].imshow(b); ax_row[1].set_title('RGB 100m')
    ax_row[2].imshow(b); ax_row[2].imshow(a, cmap='hot', alpha=0.45); ax_row[2].set_title('overlay')
    [ax.axis('off') for ax in ax_row]
plt.tight_layout(); plt.show()

### 5 · Train colorization (Pix2Pix)
Short smoke test here; full 150+50-epoch run is better done on **Kaggle** with the bundle below.

In [ ]:
#@title 5a · Smoke test (10 epochs, proves the loop works)
!python -m src.train.train_pix2pix --data data/tir2rgb --ckpt checkpoints --name tir2rgb --epochs 8 --epochs-decay 2
# back up checkpoint
!zip -qr /content/p2p_ckpt.zip checkpoints/tir2rgb && cp /content/p2p_ckpt.zip $DRIVE/

In [ ]:
#@title 5b · Bundle dataset for Kaggle full training
!zip -qr /content/kaggle_bundle.zip data/tir2rgb && cp /content/kaggle_bundle.zip $DRIVE/
print('Upload kaggle_bundle.zip as a Kaggle dataset, then in a Kaggle notebook:')
print('  !git clone', REPO)
print('  %cd tir2rgb-bah2026 && pip install -q dominate')
print('  !unzip -q /kaggle/input/<your-dataset>/kaggle_bundle.zip -d .')
print('  !python -m src.train.train_pix2pix --data data/tir2rgb --ckpt checkpoints --name tir2rgb --epochs 150 --epochs-decay 50')
print('  then zip checkpoints/tir2rgb -> download -> put in Drive as p2p_ckpt.zip')

In [ ]:
#@title 6 · Fine-tune SR model (×2 RRDBNet on TIR pairs)
!mkdir -p weights && wget -nc -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth -P weights
!python -m src.train.train_sr --data data/sr --pretrained weights/RealESRGAN_x2plus.pth \
    --out checkpoints/sr --iters 20000 --batch 4
!zip -qr /content/sr_ckpt.zip checkpoints/sr && cp /content/sr_ckpt.zip $DRIVE/

In [ ]:
#@title 7 · Restore checkpoints from Drive (fresh-runtime path)
![ -f $DRIVE/p2p_ckpt.zip ] && unzip -qo $DRIVE/p2p_ckpt.zip -d .
![ -f $DRIVE/sr_ckpt.zip ] && unzip -qo $DRIVE/sr_ckpt.zip -d .
!ls checkpoints/tir2rgb checkpoints/sr 2>/dev/null

In [ ]:
#@title 8 · Full evaluation — PSNR / SSIM / FID / water-consistency / timing
!python -m src.eval.metrics --raw data/raw \
    --sr-ckpt checkpoints/sr/sr_best.pth \
    --p2p-ckpt checkpoints/tir2rgb/latest_net_G.pth \
    --out results/eval --fid

from IPython.display import Image as I, display, Markdown
display(Markdown(open('results/eval/metrics.md').read()))
import glob
for g in sorted(glob.glob('results/eval/gallery/*.png'))[:3]:
    display(I(g, width=900))

In [ ]:
#@title 9 · Launch Gradio demo
!python -m app.demo_gradio --sr-ckpt checkpoints/sr/sr_best.pth \
    --p2p-ckpt checkpoints/tir2rgb/latest_net_G.pth --share